# NaijaCost - Part 2: Data Cleaning & Preprocessing

## Project Overview
In Part 1, we recognized that the single-row RTP metadata was insufficient and retrieved the substantial **UN World Food Programme (WFP) Nigeria Food Prices dataset** containing over 91,000 market observations across Nigerian states and markets from 2002 to present.

## Objectives for this Notebook
1. **Load and Inspect**: Load raw WFP food prices and market geolocation data.
2. **Temporal Structuring**: Parse observation dates and extract `year`, `month`, and `year_month` period indices.
3. **Unit Standardization**: Normalize diverse market units (100 KG bags, 50 KG bags, paint buckets/mudu, liters, tubers) into standardized base prices (per KG, per Liter, per Tuber, per Piece).
4. **Administrative & Geopolitical Mapping**: Map Nigerian states into standard geopolitical zones (North West, North East, North Central, South West, South East, South South).
5. **Data Quality & Anomaly Filtering**: Audit missing values, verify coordinate completeness, and remove zero/negative price anomalies.
6. **Persistence**: Export the processed dataset to `../data/processed/naijacost_cleaned_food_prices.csv` and generate an aggregated monthly summary.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configure paths
raw_dir = Path("../data/raw")
proc_dir = Path("../data/processed")
proc_dir.mkdir(parents=True, exist_ok=True)

prices_file = raw_dir / "wfp_food_prices_nga.csv"
markets_file = raw_dir / "wfp_markets_nga.csv"
clean_file = proc_dir / "naijacost_cleaned_food_prices.csv"
summary_file = proc_dir / "naijacost_monthly_summary.csv"

print(f"Loading raw data from: {prices_file}")
df_raw = pd.read_csv(prices_file)
print(f"Raw shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()


Loading raw data from: ../data/raw/wfp_food_prices_nga.csv
Raw shape: 91,216 rows, 16 columns


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Maize,51,KG,actual,Wholesale,NGN,175.92,1.54
1,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Millet,73,KG,actual,Wholesale,NGN,150.18,1.31
2,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Rice (imported),64,KG,actual,Wholesale,NGN,358.70,3.14
3,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Sorghum,65,KG,actual,Wholesale,NGN,155.61,1.36
4,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,pulses and nuts,Beans (niebe),120,KG,actual,Wholesale,NGN,196.87,1.72


## Step 1: Temporal Parsing
We convert the date string to a pandas `datetime` object and create helper features for annual, monthly, and cohort analysis.


In [3]:
df = df_raw.copy()

# Parse datetime
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['year_month'] = df['date'].dt.to_period('M').astype(str)

print("Date range:", df['date'].min().strftime('%Y-%m-%d'), "to", df['date'].max().strftime('%Y-%m-%d'))
print("Observation distribution by decade:")
print(df['year'].apply(lambda y: f"{(y//10)*10}s").value_counts().sort_index())
df


Date range: 2002-01-15 to 2026-11-15
Observation distribution by decade:
year
2000s     1110
2010s    35527
2020s    54579
Name: count, dtype: int64


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice,year,month,year_month
0,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Maize,51,KG,actual,Wholesale,NGN,175.92,1.540,2002,1,2002-01
1,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Millet,73,KG,actual,Wholesale,NGN,150.18,1.310,2002,1,2002-01
2,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Rice (imported),64,KG,actual,Wholesale,NGN,358.70,3.140,2002,1,2002-01
3,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Sorghum,65,KG,actual,Wholesale,NGN,155.61,1.360,2002,1,2002-01
4,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,pulses and nuts,Beans (niebe),120,KG,actual,Wholesale,NGN,196.87,1.720,2002,1,2002-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91211,2026-11-15,Yobe,Karasuwa,Bade (Gashua),2388,12.87,11.04,vegetables and fruits,Onions,173,0.5 KG,aggregate,Retail,NGN,250.00,0.190,2026,11,2026-11
91212,2026-11-15,Yobe,Karasuwa,Bade (Gashua),2388,12.87,11.04,vegetables and fruits,Oranges,360,400 G,aggregate,Retail,NGN,300.00,0.230,2026,11,2026-11
91213,2026-11-15,Yobe,Karasuwa,Bade (Gashua),2388,12.87,11.04,vegetables and fruits,Spinach,404,300 G,aggregate,Retail,NGN,60.00,0.045,2026,11,2026-11
91214,2026-11-15,Yobe,Karasuwa,Bade (Gashua),2388,12.87,11.04,vegetables and fruits,Tomatoes,114,0.5 KG,aggregate,Retail,NGN,200.00,0.150,2026,11,2026-11


## Step 2: Unit Normalization
In Nigerian markets, commodities are sold in diverse traditional and commercial packaging:
- **100 KG / 50 KG bags**: Wholesale grain sacks
- **2.5 KG / 2.8 KG / 2.1 KG**: Regional local measures (mudu, tiya, or paint buckets)
- **400 G / 500 G**: Canned/tinned products (e.g. powdered milk)
- **100 Tubers**: Wholesale yam heaps
- **30 pcs**: Crates of eggs

We parse the conversion factor and compute `norm_price` (price per base unit: NGN/KG, NGN/L, NGN/Tuber, NGN/Piece).


In [4]:
def parse_unit(u):
    u = str(u).strip()
    if u == 'KG':
        return 1.0, 'KG'
    elif u == 'L':
        return 1.0, 'L'
    elif u == 'Unit':
        return 1.0, 'Unit'
    elif u.endswith('KG'):
        val = float(u.replace('KG', '').strip())
        return val, 'KG'
    elif u.endswith('G'):
        val = float(u.replace('G', '').strip()) / 1000.0
        return val, 'KG'
    elif u.endswith('L'):
        val = float(u.replace('L', '').strip())
        return val, 'L'
    elif 'Tubers' in u:
        val = float(u.replace('Tubers', '').strip())
        return val, 'Tuber'
    elif 'pcs' in u:
        val = float(u.replace('pcs', '').strip())
        return val, 'Piece'
    else:
        return 1.0, u

unit_info = df['unit'].apply(parse_unit)
df['unit_factor'] = [x[0] for x in unit_info]
df['base_unit'] = [x[1] for x in unit_info]

# Compute standardized price
df['norm_price'] = (df['price'] / df['unit_factor']).round(2)
df['norm_usdprice'] = (df['usdprice'] / df['unit_factor']).round(4)

print("Unit distribution:")
print(df[['unit', 'unit_factor', 'base_unit']].drop_duplicates().head(10))


Unit distribution:
        unit  unit_factor base_unit
0         KG          1.0        KG
222   100 KG        100.0        KG
1747   50 KG         50.0        KG
2587    Unit          1.0      Unit
2592  2.6 KG          2.6        KG
2593  2.8 KG          2.8        KG
2594  2.7 KG          2.7        KG
2598       L          1.0         L
2634  2.5 KG          2.5        KG
3894   100 L        100.0         L


## Step 3: Geopolitical Zone Mapping
Nigeria's 36 states and FCT are grouped into 6 geopolitical zones. Economic dynamics (food production versus consumer demand, transport corridors) differ sharply between zones.


In [5]:
# Map states to geopolitical zones
zone_mapping = {
    'Katsina': 'North West',
    'Sokoto': 'North West',
    'Kano': 'North West',
    'Jigawa': 'North West',
    'Kaduna': 'North West',
    'Zamfara': 'North West',
    'Kebbi': 'North West',
    'Borno': 'North East',
    'Yobe': 'North East',
    'Adamawa': 'North East',
    'Gombe': 'North East',
    'Oyo': 'South West',
    'Lagos': 'South West',
    'Abia': 'South East'
}

# Add a new column for geopolitical zones based on the mapping
df['zone'] = df['admin1'].map(zone_mapping).fillna('Other')
df['commodity_clean'] = df['commodity'].str.strip()

print("Record count by Geopolitical Zone:")
print(df['zone'].value_counts())


Record count by Geopolitical Zone:
zone
North East    67923
North West    17626
South West     4628
South East     1039
Name: count, dtype: int64


## Step 4: Quality Assurance & Sanity Checks
We verify that no non-positive prices exist and check for extreme input errors.


In [6]:
initial_count = len(df)

# Ensure positive prices
df = df[df['price'] > 0].copy()

# Audit price bounds per base unit
summary_stats = df.groupby('base_unit')['norm_price'].describe()
print("Price summary by base unit:")
display(summary_stats)

print(f"Total rows retained: {len(df):,} of {initial_count:,}")


Price summary by base unit:


,count,mean,std,min,25%,50%,75%,max
base_unit,,,,,,,,
KG,77925.0,765.762879,1850.257257,0.81,169.6000,302.24,680.0000,95000.00
L,9099.0,1033.681200,1268.241408,3.66,227.5000,576.00,1333.0000,19375.00
Piece,2232.0,101.927267,79.032260,15.80,35.0000,58.33,196.4675,300.00
Tuber,472.0,515.209513,245.447632,76.67,317.4525,457.10,684.1525,1380.00
Unit,1488.0,264.635222,115.745243,96.53,185.1900,250.00,319.2500,711.39


Total rows retained: 91,216 of 91,216


## Step 5: Exporting Cleaned Datasets
We persist the cleaned dataset to `data/processed/naijacost_cleaned_food_prices.csv` and an aggregated monthly summary to `data/processed/naijacost_monthly_summary.csv` for fast exploratory and statistical downstream analysis.


In [7]:
# Save main cleaned dataset
df.to_csv(clean_file, index=False)
print(f"Cleaned dataset saved: {clean_file} ({clean_file.stat().st_size / 1e6:.2f} MB)")

# Save monthly aggregated summary
monthly_summary = df.groupby(['year_month', 'zone', 'commodity_clean', 'base_unit']).agg(
    mean_price=('norm_price', 'mean'),
    median_price=('norm_price', 'median'),
    std_price=('norm_price', 'std'),
    obs_count=('norm_price', 'count'),
    mean_usdprice=('norm_usdprice', 'mean')
).reset_index()

monthly_summary.to_csv(summary_file, index=False)
print(f"Monthly summary saved: {summary_file} ({monthly_summary.shape[0]:,} rows)")
monthly_summary.head()


Cleaned dataset saved: ../data/processed/naijacost_cleaned_food_prices.csv (16.45 MB)
Monthly summary saved: ../data/processed/naijacost_monthly_summary.csv (10,278 rows)


,year_month,zone,commodity_clean,base_unit,mean_price,median_price,std_price,obs_count,mean_usdprice
0,2002-01,North West,Beans (niebe),KG,217.970,217.970,29.839906,2,1.905
1,2002-01,North West,Maize,KG,164.635,164.635,15.959400,2,1.440
2,2002-01,North West,Millet,KG,148.565,148.565,2.283955,2,1.300
3,2002-01,North West,Rice (imported),KG,348.145,348.145,14.927024,2,3.045
4,2002-01,North West,Sorghum,KG,148.555,148.555,9.977277,2,1.300
